## Lab 4: Diffusion Models for Anomaly Detection (Tabular Data)

**Alejandro Lancho Serrano, Vanessa Gómez Verdejo, Pablo Martínez Olmos, Emilio Parrado Hernández**

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**

<img src='http://www.tsc.uc3m.es/~emipar/BBVA/INTRO/img/logo_uc3m_foot.jpg' width=400 />

In this lab, you will build an **unsupervised anomaly detection system** for credit card fraud using **diffusion models**, and compare it against classical and neural baselines.

You are given:
- basic library imports,
- the dataset.

From this point on, **you are responsible for designing, implementing, and evaluating the models**.

---

## Objectives

By the end of this lab, you should be able to:

- Apply **diffusion models** to tabular data.
- Use **denoising difficulty** as an anomaly score.
- Evaluate anomaly detectors under **extreme class imbalance**.
- Select and justify **decision thresholds**.
- Compare modern generative models with **strong baselines**.

---

## Tasks

### 1. Exploratory Data Analysis (EDA)
- Load the dataset and inspect the class imbalance.
- Visualize at least one informative feature (e.g., `Amount`) by class.
- Explain why **AUPRC** is more informative than accuracy in this setting.

*Expected outputs:* brief plots + 2–3 sentences of interpretation.

---

### 2. Diffusion Model for Tabular Data
- Implement a **forward diffusion process** for tabular vectors:
  - Choose a noise schedule (linear or cosine).
  - Implement the noising operation $ x_t $.
- Build a **denoising neural network** (MLP):
  - Input: noised data + time embedding.
  - Output: predicted noise.
- Train the model **only on normal transactions**.

*Expected outputs:* training loss curve + explanation of design choices.

---

### 3. Diffusion-Based Anomaly Scoring
Implement **at least one** of the following anomaly scores:
- **Residual-based score:** noise prediction error.
- **Reconstruction-based score:** error in reconstructed $ \hat{x}_0 $.

- Aggregate scores over multiple random timesteps.
- Explain why stochastic averaging is useful.

*Expected outputs:* function to compute scores + short explanation.

---

### 4. Evaluation (No Threshold Yet)
- Evaluate diffusion scores on **validation and test sets** using:
  - AUROC
  - AUPRC
- Plot Precision–Recall curves.
- Compare results to the random baseline.

*Expected outputs:* metrics + PR curves + interpretation.

---

### 5. Threshold Selection
Using the **validation set only**:
- Implement at least two thresholding strategies:
  - target precision or recall,
  - max-F1.
- Visualize confusion matrices for different thresholds.
- Discuss the trade-off between false positives and false negatives.

*Expected outputs:* confusion matrices + discussion.

---

### 6. Final Test Evaluation
- Fix **one threshold** chosen on validation.
- Apply it **once** to the test set.
- Report:
  - confusion matrix,
  - precision, recall, FPR, F1.

*Expected outputs:* final test results + interpretation.

---

### 7. Baseline Comparisons
Implement and evaluate **at least two baselines** (e.g.):
- **Isolation Forest**
- **Autoencoder**

Requirements:
- Train on normal samples only.
- Use the same evaluation metrics and splits.
- Produce comparable anomaly scores.

*Expected outputs:* AUROC/AUPRC on test set + PR curves.

---

### 8. Comparative Analysis
- Summarize results for:
  - Diffusion model
  - Isolation Forest
  - Autoencoder

  

*Expected outputs* must include answers to:
  - Which method performs best here?
  - Why might that be the case?
  - What does diffusion offer beyond raw performance?


## Setting Things Up <a id="setup"></a>

In [ ]:
# ======================================================
# Setup: imports, device, and reproducibility
# ======================================================

# --- Standard library ---
import os
import math
import random
import time
import json
import subprocess
import sys
from pathlib import Path
from typing import Tuple

# --- Numerical & data handling ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# --- Scikit-learn ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    confusion_matrix,
)
from sklearn.ensemble import IsolationForest

# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- Reproducibility ---
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

## Dataset <a id="data"></a>

We use the **Credit Card Fraud Detection** dataset introduced by Dal Pozzolo et al. (2015),  
containing **284,807 transactions** with a severe class imbalance  
(**fraud ≈ 0.17%**).

The dataset consists of:
- anonymized PCA features (`V1`–`V28`),
- transaction `Amount`,
- binary label `Class` (0 = normal, 1 = fraud).

---

### Automatic download options

To make the lab reproducible across environments, the notebook attempts to obtain  
`creditcard.csv` using the following **fallback chain**:

1. **Local file (fastest)**  
   - If `./data/creditcard.csv` already exists, it is loaded directly.

2. **Kaggle API (optional)**  
   - Requires a Kaggle API token at `~/.kaggle/kaggle.json`.  
   - You can generate this from *Kaggle → Account → Create API Token*.

3. **OpenML mirror (no credentials)**  
   - Downloads the same dataset via OpenML (dataset ID 1597).  
   - May be blocked on some institutional networks.

4. **Direct HTTP mirror**  
   - Downloads the dataset from a public mirror.  
   - Requires no credentials and works in most environments.

The dataset is always saved locally to: `./data/creditcard.csv`

---

### Manual fallback

If all automatic methods fail (e.g., no internet access),  
download `creditcard.csv` manually and place it in `./data/`, then re-run the notebook.

---

> **Note:**  
> All experiments in this lab use the same dataset splits and evaluation protocol,  
> regardless of which download method is used.

In [ ]:
# ======================================================
# Downloader + Loader for creditcard.csv
# Priority:
#   1) Local file
#   2) Kaggle (if credentials exist)
#   3) OpenML
#   4) Direct HTTP fallback (recommended for teaching)
# ======================================================

DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "creditcard.csv"


# ------------------------------------------------------
# Utility helpers
# ------------------------------------------------------

def have_kaggle_creds():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    return kaggle_json.exists()


def ensure_pkg(pkg):
    try:
        __import__(pkg)
        return True
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return True


# ------------------------------------------------------
# Kaggle download
# ------------------------------------------------------

def download_via_kaggle():
    """
    Requires:
      - ~/.kaggle/kaggle.json present (Kaggle API credentials)
      - Internet access
    """
    print("Attempting Kaggle download...")
    ensure_pkg("kaggle")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    os.chmod(kaggle_dir, 0o700)

    kj = kaggle_dir / "kaggle.json"
    if not kj.exists():
        print("No kaggle.json found at ~/.kaggle/kaggle.json")
        return False
    os.chmod(kj, 0o600)

    try:
        subprocess.check_call([
            sys.executable, "-m", "kaggle", "datasets", "download",
            "-d", "mlg-ulb/creditcardfraud",
            "-p", str(DATA_DIR),
            "-f", "creditcard.csv"
        ])

        if CSV_PATH.exists():
            print(f"Found {CSV_PATH}")
            return True

        # Fallback: extract from zip if needed
        import zipfile
        for z in DATA_DIR.glob("*.zip"):
            with zipfile.ZipFile(z, "r") as zf:
                if "creditcard.csv" in zf.namelist():
                    zf.extract("creditcard.csv", DATA_DIR)
                    print(f"Extracted creditcard.csv from {z}")
                    return True

        print("Kaggle download did not produce creditcard.csv")
        return False

    except subprocess.CalledProcessError as e:
        print("Kaggle download failed:", e)
        return False


# ------------------------------------------------------
# OpenML download
# ------------------------------------------------------

def download_via_openml():
    """
    Uses openml-python to fetch the dataset.
    Often blocked on institutional networks.
    """
    print("Attempting OpenML download...")
    ensure_pkg("openml")
    import openml

    try:
        # OpenML ID commonly associated with creditcard fraud
        ds = openml.datasets.get_dataset(1597)

        X, y, categorical, attribute_names = ds.get_data(
            dataset_format="dataframe",
            target=ds.default_target_attribute
        )

        df = X.copy()
        df["Class"] = y
        df.to_csv(CSV_PATH, index=False)

        print(f"Saved {CSV_PATH} via OpenML")
        return True

    except Exception as e:
        print("OpenML download failed:", e)
        return False


# ------------------------------------------------------
# Direct HTTP fallback (RECOMMENDED)
# ------------------------------------------------------

def download_via_http():
    """
    Direct public mirror.
    No credentials, works in most environments.
    """
    print("Attempting direct HTTP download...")
    url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

    try:
        df = pd.read_csv(url)
        df.to_csv(CSV_PATH, index=False)
        print(f"Saved {CSV_PATH} via HTTP")
        return True
    except Exception as e:
        print("HTTP download failed:", e)
        return False


# ------------------------------------------------------
# Unified loader
# ------------------------------------------------------

def load_creditcard_csv():
    if CSV_PATH.exists():
        print(f"Loading local {CSV_PATH}")
        return pd.read_csv(CSV_PATH)

    if have_kaggle_creds() and download_via_kaggle():
        return pd.read_csv(CSV_PATH)

    if download_via_openml():
        return pd.read_csv(CSV_PATH)

    if download_via_http():
        return pd.read_csv(CSV_PATH)

    raise FileNotFoundError(
        "Could not obtain creditcard.csv.\n"
        "Please place the file manually at ./data/creditcard.csv and re-run."
    )


# ------------------------------------------------------
# Execute
# ------------------------------------------------------

df = load_creditcard_csv()
print("Dataset shape:", df.shape)
print("First columns:", df.columns.tolist()[:10], "...")
print("Class distribution:", df["Class"].value_counts().to_dict())